In [3]:
!pip install evaluate tqdm transformers datasets

In [5]:
pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Ablation Study: Baseline Evaluation
# Đánh giá mBART50 gốc (chưa fine-tune) để so sánh với fine-tuned model

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import torch
import evaluate
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset

metric = evaluate.load("sacrebleu")

In [7]:
# 1. Load mBART50 GỐC (Baseline — chưa fine-tune)
print("Loading mBART50 baseline...")
baseline_model_name = "facebook/mbart-large-50-many-to-many-mmt"
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_model_name)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(baseline_model_name).to("cuda")
baseline_model.eval()
print("Done!")

Loading mBART50 baseline...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Done!


In [10]:
# 2. Cấu hình 3 cặp ngôn ngữ
lang_configs = [
    ("en-vi", "en_XX", "vi_VN"),
    ("en-fr", "en_XX", "fr_XX"),
    ("de-en", "de_DE", "en_XX"),
]

val_datasets = {}
for pair, src, tgt in lang_configs:
    ds = load_dataset("opus100", pair, split="validation")
    # Lấy 500 mẫu để đánh giá nhanh
    val_datasets[pair] = ds.select(range(min(500, len(ds))))
    print(f"  {pair}: {len(val_datasets[pair])} samples loaded")

README.md: 0.00B [00:00, ?B/s]

en-vi/test-00000-of-00001.parquet:   0%|          | 0.00/137k [00:00<?, ?B/s]

en-vi/train-00000-of-00001.parquet:   0%|          | 0.00/59.0M [00:00<?, ?B/s]

en-vi/validation-00000-of-00001.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/254k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  de-en: 500 samples loaded


In [14]:
# 3. Hàm dịch với mBART50 gốc
def translate_baseline(texts, src_lang, tgt_lang, model, tokenizer, batch_size=16, max_len=128):
    results = []
    tgt_lang_id = tokenizer.lang_code_to_id[tgt_lang]

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Translating {tgt_lang}"):
        batch = texts[i : i + batch_size]

        # Thêm language token vào source (mBART50 yêu cầu)
        src_lang_token = tokenizer.lang_code_to_id[src_lang]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=max_len,
            truncation=True,
            padding=True,
        )
        # Prepend source language token
        inputs["input_ids"] = torch.cat([
            torch.full((inputs["input_ids"].size(0), 1), src_lang_token, device=inputs["input_ids"].device),
            inputs["input_ids"]
        ], dim=1)
        inputs["attention_mask"] = torch.cat([
            torch.ones((inputs["attention_mask"].size(0), 1), device=inputs["attention_mask"].device),
            inputs["attention_mask"]
        ], dim=1)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        generated = model.generate(
            **inputs,
            decoder_start_token_id=tgt_lang_id,
            max_length=max_len,
            num_beams=4,
        )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)
    return results

In [15]:
# 4. Đánh giá Baseline
print("=" * 60)
print("BASELINE EVALUATION (mBART50 gốc, chưa fine-tune)")
print("=" * 60)

baseline_results = {}
for pair, src, tgt in lang_configs:
    ds = val_datasets[pair]

    # Xác định cột nguồn và đích
    src_key = "en" if pair.startswith("en") else ("de" if pair.startswith("de") else "en")
    tgt_key = "vi" if pair.endswith("vi") else ("fr" if pair.endswith("fr") else "en")

    texts = ds["translation"]
    src_texts = [t[src_key] for t in texts]
    tgt_texts = [t[tgt_key] for t in texts]

    preds = translate_baseline(src_texts, src, tgt, baseline_model, baseline_tokenizer)
    refs = [[r] for r in tgt_texts]

    result = metric.compute(predictions=preds, references=refs)
    baseline_results[pair] = result["score"]
    print(f"  {pair}: BLEU = {result['score']:.2f}")

baseline_avg = np.mean(list(baseline_results.values()))
print(f"\n  Baseline BLEU trung bình: {baseline_avg:.2f}")

BASELINE EVALUATION (mBART50 gốc, chưa fine-tune)


Translating vi_VN: 100%|██████████| 32/32 [00:16<00:00,  1.93it/s]


  en-vi: BLEU = 0.85


Translating fr_XX: 100%|██████████| 32/32 [03:31<00:00,  6.62s/it]


  en-fr: BLEU = 0.00


Translating en_XX: 100%|██████████| 32/32 [02:10<00:00,  4.08s/it]

  de-en: BLEU = 27.72

  Baseline BLEU trung bình: 9.52


In [16]:
# 5. So sánh Ablation: Baseline vs Fine-tuned
fine_tuned_results = {
    "en-vi": 20.47,
    "en-fr": 30.07,
    "de-en": 32.53,
}
fine_tuned_avg = np.mean(list(fine_tuned_results.values()))

print("\n" + "=" * 60)
print("ABLATION STUDY: Baseline vs Fine-tuned")
print("=" * 60)
print(f"\n{'Cặp ngôn ngữ':<15} {'Baseline':<12} {'Fine-tuned':<12} {'Cải thiện':<12}")
print("-" * 55)
for pair, src, tgt in lang_configs:
    p = pair
    b = baseline_results[p]
    f = fine_tuned_results[p]
    delta = f - b
    pct = (delta / b) * 100 if b > 0 else 0
    print(f"  {p:<13} {b:>8.2f}    {f:>8.2f}    +{delta:>6.2f} ({pct:>+.1f}%)")

print("-" * 55)
delta_avg = fine_tuned_avg - baseline_avg
pct_avg = (delta_avg / baseline_avg) * 100 if baseline_avg > 0 else 0
print(f"  {'Trung bình':<13} {baseline_avg:>8.2f}    {fine_tuned_avg:>8.2f}    +{delta_avg:>6.2f} ({pct_avg:>+.1f}%)")

print("\n" + "=" * 60)
print("KẾT LUẬN")
print("=" * 60)
print(f"  Fine-tuning cải thiện BLEU trung bình +{delta_avg:.2f} điểm ({pct_avg:+.1f}%)")
print(f"  Cặp en-vi hưởng lợi nhiều nhất từ fine-tuning")


ABLATION STUDY: Baseline vs Fine-tuned

Cặp ngôn ngữ    Baseline     Fine-tuned   Cải thiện   
-------------------------------------------------------
  en-vi             0.85       20.47    + 19.62 (+2315.1%)
  en-fr             0.00       30.07    + 30.07 (+829772.3%)
  de-en            27.72       32.53    +  4.81 (+17.4%)
-------------------------------------------------------
  Trung bình        9.52       27.69    + 18.17 (+190.8%)

KẾT LUẬN
  Fine-tuning cải thiện BLEU trung bình +18.17 điểm (+190.8%)
  Cặp en-vi hưởng lợi nhiều nhất từ fine-tuning
